# NGHIÊN CỨU CƠ SỞ TOÁN HỌC CỦA MÔ HÌNH TabICLv2 (2026) VÀ ỨNG DỤNG DỰ ĐOÁN NGUY CƠ SINH VIÊN BỎ HỌC

Notebook thực nghiệm này được chuyển đổi và mở rộng từ file `tabiclv2_student_dropout.py`, hiện thực hóa từng bước toàn bộ **5 trụ cột toán học của TabICLv2** (*Qu et al., 2026*) trên bộ dữ liệu `data.csv`:
1. **Đại số tuyến tính:** Phân tích hạng, phổ giá trị kỳ dị (SVD), **Repeated Feature Grouping** $\text{Group}_j = (j, (j+1)\bmod m, (j+3)\bmod m)$, **Target-Aware Embedding (TAE)** và phép nhân ma trận $Q, K, V$.
2. **Giải tích:** Hàm **Softmax**, cơ chế **QASSMax** (*Query-Aware Scalable Softmax*), ma trận Jacobian $\mathbf{J}_{\sigma}(\mathbf{z}) = \operatorname{diag}(\hat{\mathbf{p}}) - \hat{\mathbf{p}}\hat{\mathbf{p}}^\top$ và Gradient hàm mất mát **Cross-Entropy** $\nabla_{\mathbf{z}}\mathcal{L} = \hat{\mathbf{p}} - \mathbf{y}$.
3. **Xác suất thống kê:** Phân phối xác suất hậu nghiệm có điều kiện $P(Y_{\text{test}} \mid \mathbf{x}_{\text{test}}, \mathcal{D}_{\text{train}})$, Shannon Entropy và quyết định theo **Rủi ro Bayes** ($\tau^* = \frac{C_{\text{FP}}}{C_{\text{FP}} + C_{\text{FN}}}$).
4. **Tối ưu hóa:** **Muon Optimizer** (*MomentUm Orthogonalized by Newton-Schulz*) với phép lặp đa thức bậc 5 xấp xỉ thừa số cực trực giao $\mathbf{U}_t \mathbf{V}_t^\top$.
5. **In-Context Learning (ICL):** Suy luận dự đoán nguy cơ bỏ học trực tiếp từ tập ngữ cảnh $\mathcal{D}_{\text{train}}$ mà **không cập nhật trọng số** ($\Delta \Theta^* = \mathbf{0}$) trên tập kiểm thử.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import math
import numpy as np
import pandas as pd

np.random.seed(42)

# Đọc bộ dữ liệu sinh viên bỏ học: dò data.csv ở thư mục hiện tại và các thư mục cha
# (notebook có thể nằm ở gốc dự án, notebooks/ hay notebooks/ban_khac/)
CANDIDATES = [os.path.join(up, sub) for up in ("", "..", "../..", "../../..") for sub in ("data/data.csv", "data.csv")]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Không thấy data.csv. Thư mục hiện tại: {os.getcwd()}")
df = pd.read_csv(DATA_PATH, sep=";")
df.columns = [c.strip() for c in df.columns]  # Loại bỏ ký tự tab thừa ở cột thứ 5

print(f"Đã đọc: {os.path.abspath(DATA_PATH)}")
print(f"Kích thước bảng dữ liệu: {df.shape[0]} sinh viên x {df.shape[1]} cột (36 đặc trưng + 1 cột Target)")
print(f"Số giá trị khuyết thiếu : {df.isnull().sum().sum()}")
print(f"5 dòng dữ liệu đầu tiên (5 cột đầu + Target):")
print(df.iloc[:5, [0, 1, 2, 3, 4, -1]].to_string())

## Bước 1: Phân tích Đại số Tuyến tính (SVD, Hạng ma trận) & Xác suất Tiên nghiệm trên `data.csv`
- Kiểm tra hạng của ma trận đặc trưng $\mathbf{X} \in \mathbb{R}^{4424 \times 36}$, số điều kiện phổ $\kappa_2(\mathbf{X}_{\text{std}}) = \frac{\sigma_{\max}}{\sigma_{\min}}$.
- Tính phân phối xác suất tiên nghiệm $P(Y = c)$, độ bất định Shannon Entropy $H(Y) = -\sum_c p_c \ln p_c$, và top đặc trưng tương quan mạnh nhất với `Dropout`.

In [2]:
label_map = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
X_raw = df.drop(columns=["Target"]).values.astype(np.float64)
y_all = df["Target"].map(label_map).values.astype(np.int64)

# 1. Phân tích SVD và số điều kiện trên ma trận chuẩn hóa
X_std_all = (X_raw - X_raw.mean(axis=0)) / (X_raw.std(axis=0) + 1e-8)
u, s, vt = np.linalg.svd(X_std_all, full_matrices=False)
cond_num = s[0] / s[-1]
var_exp = np.cumsum(s**2) / np.sum(s**2)

print(f"1. Hạng ma trận rank(X)         : {np.linalg.matrix_rank(X_raw)} / {X_raw.shape[1]}")
print(f"2. Giá trị kỳ dị lớn/nhỏ nhất  : sigma_max = {s[0]:.2f}, sigma_min = {s[-1]:.2f}")
print(f"3. Số điều kiện phổ kappa_2(X) : {cond_num:.4f}")
print(f"4. Phương sai tích lũy SVD     : Top-5={var_exp[4]*100:.2f}%, Top-10={var_exp[9]*100:.2f}%, Top-20={var_exp[19]*100:.2f}%")

# 2. Phân phối xác suất tiên nghiệm và Shannon Entropy
probs = df["Target"].value_counts(normalize=True)
entropy = -np.sum(probs * np.log(probs))
print(f"\n5. Phân phối Target            : {probs.round(4).to_dict()}")
print(f"6. Shannon Entropy H(Y)        : {entropy:.4f} nats (Max ln(3) = {np.log(3):.4f} nats)")

# 3. Top 8 đặc trưng có tương quan Pearson mạnh nhất với nguy cơ Dropout
is_dropout = (y_all == 0).astype(float)
corrs = pd.Series(
    [np.corrcoef(X_raw[:, j], is_dropout)[0, 1] for j in range(X_raw.shape[1])],
    index=df.columns[:-1]
).sort_values(key=abs, ascending=False)
print("\n7. Top 8 đặc trưng tương quan mạnh nhất với Dropout:")
for feat, r_val in corrs.head(8).items():
    print(f"   {feat:40s}: r = {r_val:+.4f}")

1. Hạng ma trận rank(X)         : 36 / 36
2. Giá trị kỳ dị lớn/nhỏ nhất  : sigma_max = 167.74, sigma_min = 9.60
3. Số điều kiện phổ kappa_2(X) : 17.4781
4. Phương sai tích lũy SVD     : Top-5=44.26%, Top-10=64.65%, Top-20=88.12%

5. Phân phối Target            : {'Graduate': 0.4993, 'Dropout': 0.3212, 'Enrolled': 0.1795}
6. Shannon Entropy H(Y)        : 1.0199 nats (Max ln(3) = 1.0986 nats)

7. Top 8 đặc trưng tương quan mạnh nhất với Dropout:
   Curricular units 2nd sem (grade)        : r = -0.5718
   Curricular units 2nd sem (approved)     : r = -0.5695
   Curricular units 1st sem (grade)        : r = -0.4807
   Curricular units 1st sem (approved)     : r = -0.4791
   Tuition fees up to date                 : r = -0.4291
   Age at enrollment                       : r = +0.2542
   Scholarship holder                      : r = -0.2454
   Debtor                                  : r = +0.2294


## Bước 2: Chia tập Ngữ cảnh ($\mathcal{D}_{\text{train}}$) & Tập Truy vấn ($\mathcal{D}_{\text{test}}$) và Nhóm Đặc trưng Dịch vòng (*Repeated Feature Grouping*)
Trong TabICLv2, mỗi cột $j \in \{0, \dots, m-1\}$ ($m=36$) được gom nhóm theo mẫu dịch vòng $(0, 1, 3) \pmod m$:
$$\mathbf{g}_{i,j} = \big[x_{i,j},\; x_{i,(j+1)\bmod m},\; x_{i,(j+3)\bmod m}\big]^\top \in \mathbb{R}^3, \qquad \mathbf{E}_1[i,j] = \tanh(\mathbf{W}_{\text{grp}}[j]\mathbf{g}_{i,j} + \mathbf{b}_{\text{grp}}[j])$$

In [3]:
# Chia Stratified 80% làm tập ngữ cảnh (D_train) và 20% làm tập kiểm thử (D_test)
np.random.seed(42)
train_idx, test_idx = [], []
for c in [0, 1, 2]:
    c_idx = np.where(y_all == c)[0]
    np.random.shuffle(c_idx)
    split = int(0.8 * len(c_idx))
    train_idx.extend(c_idx[:split])
    test_idx.extend(c_idx[split:])
train_idx, test_idx = np.array(train_idx), np.array(test_idx)

X_train_raw, y_train = X_raw[train_idx], y_all[train_idx]
X_test_raw, y_test = X_raw[test_idx], y_all[test_idx]

# Chuẩn hóa Z-score theo thống kê của tập ngữ cảnh D_train
mean = X_train_raw.mean(axis=0, keepdims=True)
std = X_train_raw.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

def repeated_feature_grouping(X: np.ndarray, W_grp: np.ndarray, b_grp: np.ndarray) -> np.ndarray:
    '''
    Thực hiện Repeated Feature Grouping theo offset (0, 1, 3) mod m.
    X: [N, m], W_grp: [m, 3, d_col], b_grp: [m, d_col] -> Trả về E_1: [N, m, d_col]
    '''
    x_0 = X
    x_1 = np.roll(X, shift=-1, axis=1)
    x_3 = np.roll(X, shift=-3, axis=1)
    G = np.stack([x_0, x_1, x_3], axis=-1)  # [N, m, 3]
    E_1 = np.einsum("nmk,mkd->nmd", G, W_grp) + b_grp[None, :, :]
    return np.tanh(E_1)

N_train, m = X_train.shape
C = 3
d_col = 4
D = m * d_col  # 36 * 4 = 144

W_grp = np.random.randn(m, 3, d_col) * 0.25
b_grp = np.zeros((m, d_col))

E1_train = repeated_feature_grouping(X_train, W_grp, b_grp)
E1_test = repeated_feature_grouping(X_test, W_grp, b_grp)
H_train = E1_train.reshape(N_train, D)
H_test = E1_test.reshape(len(X_test), D)

print(f"Số mẫu ngữ cảnh (N_train) : {N_train} | Số mẫu kiểm thử (N_test) : {len(X_test)}")
print(f"Tensor sau Repeated Feature Grouping E1_train : {E1_train.shape}")
print(f"Ma trận biểu diễn dòng sinh viên H_train      : {H_train.shape}")

Số mẫu ngữ cảnh (N_train) : 3538 | Số mẫu kiểm thử (N_test) : 886
Tensor sau Repeated Feature Grouping E1_train : (3538, 36, 4)
Ma trận biểu diễn dòng sinh viên H_train      : (3538, 144)


## Bước 3: Tối ưu hóa Muon Optimizer (Trực giao hóa Đa thức Bậc 5 Newton-Schulz)
Thay vì tính SVD trực tiếp $\mathbf{M}_t = \mathbf{U}_t \boldsymbol{\Sigma}_t \mathbf{V}_t^\top$, Muon chuẩn hóa chuẩn Frobenius $\mathbf{X}_0 = \frac{\mathbf{M}_t}{\|\mathbf{M}_t\|_F + \epsilon}$ và lặp $K=5$ bước đa thức bậc 5 với $(a, b, c) = (3.4445, -4.7750, 2.0315)$:
$$\mathbf{X}_{k+1} = a\mathbf{X}_k + b(\mathbf{X}_k \mathbf{X}_k^\top)\mathbf{X}_k + c(\mathbf{X}_k \mathbf{X}_k^\top)^2\mathbf{X}_k \xrightarrow{k=5} \mathbf{U}_t \mathbf{V}_t^\top$$

In [4]:
def newton_schulz_orthogonalize(G: np.ndarray, steps: int = 5, eps: float = 1e-7) -> np.ndarray:
    '''
    Xấp xỉ thừa số cực trực giao U @ V^T bằng phép lặp đa thức bậc 5 Newton-Schulz.
    '''
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.astype(np.float64)
    X = X / (np.linalg.norm(X, ord="fro") + eps)
    transposed = False
    if X.shape[0] > X.shape[1]:
        X = X.T
        transposed = True
    for _ in range(steps):
        A = X @ X.T
        X = a * X + (b * A + c * (A @ A)) @ X
    if transposed:
        X = X.T
    return X

# Kiểm chứng phổ giá trị kỳ dị trước và sau khi qua 5 bước Newton-Schulz
sample_grad = np.random.randn(144, 48) @ np.diag(np.linspace(0.01, 50.0, 48))
s_before = np.linalg.svd(sample_grad, compute_uv=False)
ortho_grad = newton_schulz_orthogonalize(sample_grad, steps=5)
s_after = np.linalg.svd(ortho_grad, compute_uv=False)

print(f"Trước Newton-Schulz : sigma_min = {s_before.min():.4f}, sigma_max = {s_before.max():.4f}, tỷ số kappa = {s_before.max()/s_before.min():.1f}")
print(f"Sau 5 bước Muon NS  : sigma_min = {s_after.min():.4f}, sigma_max = {s_after.max():.4f}, trung bình = {s_after.mean():.4f} (xấp xỉ 1.0)")

Trước Newton-Schulz : sigma_min = 0.0991, sigma_max = 681.2766, tỷ số kappa = 6875.0
Sau 5 bước Muon NS  : sigma_min = 0.0198, sigma_max = 1.1806, trung bình = 0.8408 (xấp xỉ 1.0)


## Bước 4: Giải tích QASSMax Attention, Gradient Cross-Entropy & Huấn luyện Episodic với Muon
- **Cơ chế QASSMax:** $\tilde{\mathbf{Q}} = \mathbf{Q} \odot \frac{\ln N_{\text{ctx}}}{\sqrt{d_k}} \odot \big(1 + \tanh(\mathbf{Q}\mathbf{W}_{\text{gate}})\big)$.
- **Gradient hàm mất mát Cross-Entropy:** $\nabla_{\mathbf{Z}}\mathcal{L} = \frac{1}{N_q}(\hat{\mathbf{P}} - \mathbf{Y})$.

In [5]:
def softmax(z: np.ndarray, axis: int = -1) -> np.ndarray:
    z_max = np.max(z, axis=axis, keepdims=True)
    exp_z = np.exp(z - z_max)
    return exp_z / np.sum(exp_z, axis=axis, keepdims=True)

np.random.seed(42)
d_k = 48
W_QK = np.eye(D, d_k) + np.random.randn(D, d_k) * 0.05
W_gate = np.random.randn(d_k, d_k) * 0.05
W_V = np.random.randn(D, C) * 0.05
W_res = np.zeros((D, C))
alpha_tae = 2.5

buf_W_QK = np.zeros_like(W_QK)
buf_W_res = np.zeros_like(W_res)
mu = 0.95
lr_muon = 0.08
Y_train_oh = np.eye(C)[y_train]

history_loss, history_acc = [], []
print("=== BẮT ĐẦU HUẤN LUYỆN EPISODIC IN-CONTEXT LEARNING VỚI MUON OPTIMIZER ===")
for epoch in range(1, 61):
    perm = np.random.permutation(N_train)
    ctx_i, qry_i = perm[:800], perm[800:1200]

    H_ctx, Y_ctx = H_train[ctx_i], Y_train_oh[ctx_i]
    H_qry, Y_qry = H_train[qry_i], Y_train_oh[qry_i]

    # 1. Phép chiếu tuyến tính Q, K và điều biến QASSMax
    Q = H_qry @ W_QK
    K = H_ctx @ W_QK
    base_scale = math.log(len(ctx_i)) / math.sqrt(d_k)
    gate = 1.0 + np.tanh(Q @ W_gate)
    Q_tilde = Q * base_scale * gate

    # 2. Ma trận Attention & Tổ hợp giá trị có nhúng nhãn (TAE)
    S = Q_tilde @ K.T
    A = softmax(S, axis=-1)
    V_ctx = alpha_tae * Y_ctx + (H_ctx @ W_V)
    Z_qry = (A @ V_ctx) + (H_qry @ W_res)
    P_qry = softmax(Z_qry, axis=-1)

    # 3. Hàm mất mát Cross-Entropy
    loss = -np.mean(np.sum(Y_qry * np.log(P_qry + 1e-12), axis=-1))
    acc_tr = np.mean(np.argmax(P_qry, axis=-1) == y_train[qry_i]) * 100
    history_loss.append(loss)
    history_acc.append(acc_tr)

    # 4. Giải tích Gradient (Chain Rule qua Cross-Entropy & Softmax Jacobian)
    dZ = (P_qry - Y_qry) / len(qry_i)
    G_W_res = H_qry.T @ dZ

    dA = dZ @ V_ctx.T
    dS = A * (dA - np.sum(dA * A, axis=-1, keepdims=True))
    dQ_tilde = dS @ K
    dK = dS.T @ Q_tilde
    dQ = dQ_tilde * base_scale * gate
    G_W_QK = (H_qry.T @ dQ) + (H_ctx.T @ dK)

    # 5. Cập nhật trọng số bằng Muon Optimizer (Newton-Schulz 5 bước)
    buf_W_res = mu * buf_W_res + G_W_res
    O_res = newton_schulz_orthogonalize(G_W_res + mu * buf_W_res, steps=5)
    W_res -= lr_muon * math.sqrt(max(1.0, W_res.shape[0] / W_res.shape[1])) * O_res

    buf_W_QK = mu * buf_W_QK + G_W_QK
    O_QK = newton_schulz_orthogonalize(G_W_QK + mu * buf_W_QK, steps=5)
    W_QK -= lr_muon * math.sqrt(max(1.0, W_QK.shape[0] / W_QK.shape[1])) * O_QK

    if epoch in (1, 15, 30, 45, 60):
        print(f"  Epoch {epoch:2d}/60 | Cross-Entropy Loss = {loss:.4f} | Query Accuracy = {acc_tr:.2f}%")

=== BẮT ĐẦU HUẤN LUYỆN EPISODIC IN-CONTEXT LEARNING VỚI MUON OPTIMIZER ===
  Epoch  1/60 | Cross-Entropy Loss = 0.9290 | Query Accuracy = 61.75%
  Epoch 15/60 | Cross-Entropy Loss = 0.7189 | Query Accuracy = 72.75%
  Epoch 30/60 | Cross-Entropy Loss = 0.6500 | Query Accuracy = 74.75%
  Epoch 45/60 | Cross-Entropy Loss = 0.7215 | Query Accuracy = 72.00%
  Epoch 60/60 | Cross-Entropy Loss = 0.8212 | Query Accuracy = 71.00%


## Bước 5: Suy luận In-Context Learning (Không cập nhật trọng số) & Phân tích Quyết định Rủi ro Bayes trên Tập Kiểm thử ($N_{\text{test}} = 886$)
Toàn bộ tập huấn luyện $\mathcal{D}_{\text{train}}$ ($3538$ sinh viên) được dùng làm ngữ cảnh $(\mathbf{K}_{\text{full}}, \mathbf{V}_{\text{full}})$ để dự đoán trực tiếp xác suất bỏ học cho $886$ sinh viên mới mà không thực hiện thêm bất kỳ bước cập nhật trọng số nào.

In [6]:
# Suy luận In-Context Learning trong 1 lượt truyền thẳng (Single Forward Pass)
Q_test = H_test @ W_QK
K_full = H_train @ W_QK
base_scale_test = math.log(N_train) / math.sqrt(d_k)
gate_test = 1.0 + np.tanh(Q_test @ W_gate)
Q_test_tilde = Q_test * base_scale_test * gate_test

A_test = softmax(Q_test_tilde @ K_full.T, axis=-1)
V_full = alpha_tae * Y_train_oh + (H_train @ W_V)
Z_test = (A_test @ V_full) + (H_test @ W_res)
P_test = softmax(Z_test, axis=-1)
preds_test = np.argmax(P_test, axis=-1)

acc_test = np.mean(preds_test == y_test) * 100
y_true_drop = (y_test == 0).astype(int)
p_drop = P_test[:, 0]

# Tính ROC-AUC cho lớp Dropout
pos_scores = p_drop[y_true_drop == 1]
neg_scores = p_drop[y_true_drop == 0]
auc_dropout = np.mean(pos_scores[:, None] > neg_scores[None, :]) + 0.5 * np.mean(pos_scores[:, None] == neg_scores[None, :])

print(f"=== KẾT QUẢ SUY LUẬN IN-CONTEXT LEARNING TRÊN TẬP TEST (N_test = {len(y_test)}) ===")
print(f"- Độ chính xác đa lớp (3-class Accuracy)      : {acc_test:.2f}%")
print(f"- Diện tích dưới đường cong ROC (Dropout AUC) : {auc_dropout:.4f}\n")

# So sánh ngưỡng quyết định mặc định (0.50) và ngưỡng tối ưu rủi ro Bayes (0.25)
rows_eval = []
for tau, desc in [(0.50, "Ngưỡng chuẩn (tau = 0.50)"), (0.25, "Ngưỡng Rủi ro Bayes (tau* = 0.25, C_FN = 3*C_FP)")]:
    pred_bin = (p_drop >= tau).astype(int)
    tp = int(np.sum((pred_bin == 1) & (y_true_drop == 1)))
    fp = int(np.sum((pred_bin == 1) & (y_true_drop == 0)))
    fn = int(np.sum((pred_bin == 0) & (y_true_drop == 1)))
    prec = tp / max(1, tp + fp) * 100
    rec = tp / max(1, tp + fn) * 100
    f1 = 2 * prec * rec / max(1e-8, prec + rec)
    rows_eval.append({"Chiến lược ngưỡng": desc, "TP": tp, "FP": fp, "FN": fn, "Precision (%)": round(prec, 2), "Recall (%)": round(rec, 2), "F1-Score (%)": round(f1, 2)})

eval_df = pd.DataFrame(rows_eval)
print(eval_df.to_string(index=False))

# Hiển thị xác suất dự đoán cho 8 sinh viên đầu tiên trong tập Test
label_inv = {0: "Dropout", 1: "Enrolled", 2: "Graduate"}
sample_res = pd.DataFrame({
    "Nhãn thực tế": [label_inv[y] for y in y_test[:8]],
    "Dự đoán ICL": [label_inv[y] for y in preds_test[:8]],
    "P(Dropout)": np.round(P_test[:8, 0], 4),
    "P(Enrolled)": np.round(P_test[:8, 1], 4),
    "P(Graduate)": np.round(P_test[:8, 2], 4),
})
print("\n=== MINH HỌA PHÂN PHỐI XÁC SUẤT HẬU NGHIỆM CHO 8 SINH VIÊN TEST ĐẦU TIÊN ===")
print(sample_res.to_string(index=False))

=== KẾT QUẢ SUY LUẬN IN-CONTEXT LEARNING TRÊN TẬP TEST (N_test = 886) ===
- Độ chính xác đa lớp (3-class Accuracy)      : 75.96%
- Diện tích dưới đường cong ROC (Dropout AUC) : 0.8837

                               Chiến lược ngưỡng  TP  FP  FN  Precision (%)  Recall (%)  F1-Score (%)
                       Ngưỡng chuẩn (tau = 0.50) 204  37  81          84.65       71.58         77.57
Ngưỡng Rủi ro Bayes (tau* = 0.25, C_FN = 3*C_FP) 224  74  61          75.17       78.60         76.84

=== MINH HỌA PHÂN PHỐI XÁC SUẤT HẬU NGHIỆM CHO 8 SINH VIÊN TEST ĐẦU TIÊN ===
Nhãn thực tế Dự đoán ICL  P(Dropout)  P(Enrolled)  P(Graduate)
     Dropout     Dropout      0.8164       0.1175       0.0661
     Dropout     Dropout      0.3857       0.3500       0.2644
     Dropout     Dropout      0.9341       0.0658       0.0001
     Dropout     Dropout      0.9436       0.0558       0.0006
     Dropout    Graduate      0.0050       0.1264       0.8686
     Dropout     Dropout      0.8433       0.1566    